In [1]:
!pip install -q ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.1 MB/s eta 0:00:00a 0:00:01


In [2]:
import yaml, shutil

# Just copy the yaml file only — not the whole dataset
shutil.copy(
    '/kaggle/input/datasets/vishweshvishwakarma/geo-dataset/yolo_split/dataset.yaml',
    '/kaggle/working/dataset.yaml'
)

# Fix the yaml to point back to input folder
yaml_path = '/kaggle/working/dataset.yaml'
with open(yaml_path, 'r') as f:
    data = yaml.safe_load(f)

data['path']  = '/kaggle/input/datasets/vishweshvishwakarma/geo-dataset/yolo_split/'
data['train'] = 'train/images'
data['val']   = 'val/images'
data['test']  = 'test/images'

with open(yaml_path, 'w') as f:
    yaml.dump(data, f)
print("Done!")
print(open(yaml_path).read())

Done!
names:
  0: BuiltUp_Roof1
  1: BuiltUp_Roof2
  2: BuiltUp_Roof3
  3: BuiltUp_Roof4
  4: Road_1
  5: Road_3
  6: Road_4
  7: Road_5
  8: Road_6
  9: Utility
  10: Water
  11: Bridge
  12: Railway
nc: 13
path: /kaggle/input/datasets/vishweshvishwakarma/geo-dataset/yolo_split/
test: test/images
train: train/images
val: val/images



In [3]:
from ultralytics import YOLO

model = YOLO('yolo26m-seg.pt')  # medium model for better attention/context

results = model.train(
    data='/kaggle/working/dataset.yaml',
    task='segment',
    cache=False,
    epochs=30,
    imgsz=768,       
    batch=8,           # smaller batch for larger tiles on T4 x2
    device=[0, 1],
    optimizer='AdamW',
    lr0=0.001,         # fresh start with medium model
    lrf=0.01,
    cos_lr=True,
    warmup_epochs=3,
    mosaic=0.0,
    fliplr=0.5,
    flipud=0.5,
    patience=15,
    save=True,
    save_period=5,
    project='/kaggle/working/yolo26_m_seg_project',
    name='run_v1',
    exist_ok=True,
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.25 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
                                                       CUDA:1 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/dataset.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.5, format=torchscript, fraction=

In [4]:
import shutil
shutil.copy(
    '/kaggle/working/yolo26_m_seg_project/run_v1/weights/best.pt',
    '/kaggle/working/best_v1.pt'
)
print("Saved!")

Saved!


In [5]:
import shutil

shutil.make_archive(
    '/kaggle/working/yolo_run',  # output zip name
    'zip',
    '/kaggle/working/yolo26_m_seg_project/run_v1'   # folder to zip
)
print("Done!")

Done!
